Import the Mlp class and the Protein Strings

In [1]:
from protgen.data import ProteinDataset
from protgen.models.mlp import MLP
import torch 
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader

Load the Data 

In [2]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")



Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])


Create the model and test it with a small input 

In [14]:
mlp = MLP()

X = torch.LongTensor([[0, 0, 1], [0, 1, 2], [1, 2, 3]])
Y = torch.LongTensor([1,2,3])

logits = mlp(X)
targets = Y

Loss = F.cross_entropy(logits, targets)
print(f"Logits: {logits}")
print(f"Targets: {targets}")
print(f"Loss: {Loss}")


Logits: tensor([[-1.1899e-01, -7.3317e-02,  1.1609e-02, -3.0125e-02, -3.6817e-02,
         -2.3869e-04, -1.1433e-02, -1.9069e-01,  2.8905e-01,  4.6878e-02,
          5.6827e-03,  4.9592e-02, -2.9338e-01, -2.6577e-02,  1.0633e-01,
         -1.0640e-01, -5.0605e-01, -5.8891e-02, -6.4671e-02, -2.0423e-01,
         -1.2818e-01, -2.9533e-01,  4.1163e-01],
        [-1.1136e-01,  1.0999e-01,  9.6199e-02, -3.0028e-01, -5.8164e-02,
          2.0563e-01,  1.2336e-01,  7.5428e-02,  1.2414e-01,  2.0379e-01,
          2.4346e-01,  2.0437e-01, -2.1356e-01, -6.7543e-02,  2.4974e-01,
         -2.7697e-01, -2.6529e-01,  1.5097e-01,  1.8056e-01, -3.1965e-01,
         -2.6445e-01, -4.0584e-01, -1.4319e-02],
        [-5.2877e-02,  1.5958e-01,  1.6459e-01, -4.2312e-01, -3.5947e-02,
         -2.8389e-01,  5.0867e-02, -1.6374e-01,  1.2521e-01,  5.3711e-03,
          1.5111e-01, -2.5450e-02, -1.4271e-01, -9.5939e-02,  1.0309e-02,
          2.7667e-02, -8.6931e-01, -4.2329e-02,  3.1343e-01, -1.4930e-01,
      

Optimizer Function

In [15]:
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

Overfitting over the small input test

In [16]:
for epoch in range(1000):
    logits = mlp(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 3.2437
Epoch 100: Loss = 0.0883
Epoch 200: Loss = 0.0150
Epoch 300: Loss = 0.0065
Epoch 400: Loss = 0.0037
Epoch 500: Loss = 0.0024
Epoch 600: Loss = 0.0017
Epoch 700: Loss = 0.0012
Epoch 800: Loss = 0.0010
Epoch 900: Loss = 0.0008


Format the tranning data and create the 3 Tokens sequences

Traning Loop

In [ ]:
# number of training passses
num_epochs = 1000
# store loss and accuracy for information
train_losses, val_losses, train_accs, val_accs = [], [], [], []


# training loop
for e in range(num_epochs):
    # get training input and expected output as torch Variables and make sure type is correct
    tr_input = torch.from_numpy(X_tr)
    tr_targets = torch.from_numpy(onehot(y_tr, num_output)).float()

    # zeroize accumulated gradients in parameters
    optimizer.zero_grad()
    # predict by running forward pass
    tr_output = net(tr_input)
    # compute cross entropy loss
    tr_loss = cross_entropy(tr_output, tr_targets)
    # compute gradients given loss
    tr_loss.backward()
    # update the parameters given the computed gradients
    optimizer.step()
    train_acc = accuracy(tr_output, tr_targets)

    # store training loss
    train_losses.append(tr_loss.data.numpy())
    train_accs.append(train_acc)

    # get validation input and expected output as torch Variables and make sure type is correct
    val_input = torch.from_numpy(X_val)
    val_targets = torch.from_numpy(onehot(y_val, num_output)).float()

    # predict with validation input
    val_output = net(val_input)
    # compute loss and accuracy
    val_loss = cross_entropy(val_output, val_targets)
    val_acc = accuracy(val_output, val_targets)

    # store loss and accuracy
    val_losses.append(val_loss.data.numpy())
    val_accs.append(val_acc.data.numpy())

    if e % 100 == 0:
        print("Epoch %i, "
              "Train Cost: %0.3f"
              "\tVal Cost: %0.3f"
              "\t Val acc: %0.3f" % (e,
                                     train_losses[-1],
                                     val_losses[-1],
                                     val_accs[-1]))
